In [1]:
!pip install openpyxl

In [3]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

In [5]:
marker_df = pd.read_excel("/Users/hotboytraidat/Desktop/project data/Cell_marker_All.xlsx")
interaction_df = pd.read_csv("/Users/hotboytraidat/Desktop/project data/data raw/interaction_input.csv")
gene_df = pd.read_csv("/Users/hotboytraidat/Desktop/project data/data raw/gene_input.csv")
complex_df = pd.read_csv("/Users/hotboytraidat/Desktop/project data/data raw/complex_input.csv")

In [9]:
for col in ["species", "cell_name", "cell_type", "Symbol"]:
    marker_df[col] = marker_df[col].fillna("").astype(str).str.strip()

marker_df = marker_df[
    (marker_df["species"] != "") &
    (marker_df["Symbol"] != "")
].copy()

marker_df["cell_label"] = marker_df["cell_name"]
marker_df.loc[marker_df["cell_label"] == "", "cell_label"] = marker_df.loc[marker_df["cell_label"] == "", "cell_type"]
marker_df = marker_df[marker_df["cell_label"] != ""].copy()

# chỉ lấy Human cho bản này
marker_df = marker_df[marker_df["species"].str.lower() == "human"].copy()

print("Marker rows:", len(marker_df))
print("Unique human cells:", marker_df["cell_label"].nunique())
marker_df.head()

Marker rows: 53978
Unique human cells: 1596


,species,tissue_class,tissue_type,uberonongology_id,cancer_type,cell_type,cell_name,cellontology_id,marker,Symbol,...,Genetype,Genename,UNIPROTID,technology_seq,marker_source,PMID,Title,journal,year,cell_label
0,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,MERTK,MERTK,...,protein_coding,"MER proto-oncogene, tyrosine kinase",Q12866,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
1,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD16,FCGR3A,...,protein_coding,Fc fragment of IgG receptor IIIb,O75015,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
2,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD206,MRC1,...,protein_coding,mannose receptor C-type 1,P22897,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
3,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CRIg,VSIG4,...,protein_coding,V-set and immunoglobulin domain containing 4,Q9Y279,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
4,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD163,CD163,...,protein_coding,CD163 molecule,Q86VB7,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage


In [11]:
for col in ["uniprot", "hgnc_symbol"]:
    gene_df[col] = gene_df[col].fillna("").astype(str).str.strip()

gene_df = gene_df[(gene_df["uniprot"] != "") & (gene_df["hgnc_symbol"] != "")].copy()

uniprot_to_gene = dict(zip(gene_df["uniprot"], gene_df["hgnc_symbol"]))

print("Gene mappings:", len(uniprot_to_gene))
gene_df.head()

Gene mappings: 1561


,gene_name,uniprot,hgnc_symbol,ensembl
0,SULT1A1,P50225,SULT1A1,ENSG00000196502
1,UBASH3B,Q8TF42,UBASH3B,ENSG00000154127
2,SRD5A3,Q9H8P0,SRD5A3,ENSG00000128039
3,SULT2A1,Q06520,SULT2A1,ENSG00000105398
4,SRD5A2,P31213,SRD5A2,ENSG00000277893


In [13]:
complex_cols = ["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4", "uniprot_5"]

for col in ["complex_name"] + complex_cols:
    complex_df[col] = complex_df[col].fillna("").astype(str).str.strip()

complex_to_genes = {}

for _, row in complex_df.iterrows():
    complex_name = row["complex_name"]
    genes = []

    for c in complex_cols:
        u = row[c]
        if u and u in uniprot_to_gene:
            genes.append(uniprot_to_gene[u])

    genes = sorted(list(set([g for g in genes if g != ""])))
    complex_to_genes[complex_name] = genes

print("Complex mappings:", len(complex_to_genes))
list(complex_to_genes.items())[:3]

Complex mappings: 362


[('Dehydroepiandrosterone_bySTS', ['UBASH3B']),
 ('DHEAsulfate_bySULT2B', ['SULT1A1']),
 ('Dihydrotestosterone_bySRD5A3', ['SRD5A3'])]

In [15]:
def partner_to_genes(partner):
    partner = str(partner).strip()

    if partner == "" or partner.lower() == "nan":
        return []

    # partner là 1 uniprot ID
    if partner in uniprot_to_gene:
        return [uniprot_to_gene[partner]]

    # partner là complex name
    if partner in complex_to_genes:
        return complex_to_genes[partner]

    return []

In [17]:
interaction_df["partner_a"] = interaction_df["partner_a"].fillna("").astype(str).str.strip()
interaction_df["partner_b"] = interaction_df["partner_b"].fillna("").astype(str).str.strip()
interaction_df["classification"] = interaction_df["classification"].fillna("").astype(str).str.strip()
interaction_df["interactors"] = interaction_df["interactors"].fillna("").astype(str).str.strip()

interaction_df["genes_a"] = interaction_df["partner_a"].apply(partner_to_genes)
interaction_df["genes_b"] = interaction_df["partner_b"].apply(partner_to_genes)

# chỉ giữ interaction có map được ít nhất 1 gene ở mỗi phía
interaction_clean = interaction_df[
    (interaction_df["genes_a"].apply(len) > 0) &
    (interaction_df["genes_b"].apply(len) > 0)
].copy()

print("Mapped interactions:", len(interaction_clean))
interaction_clean[["partner_a", "partner_b", "genes_a", "genes_b", "classification", "interactors"]].head()

Mapped interactions: 2911


,partner_a,partner_b,genes_a,genes_b,classification,interactors
0,P12830,integrin_a2b1_complex,[CDH1],"[ITGA2, ITGB1]",Adhesion by Cadherin,CDH1-ITGA2+ITGB1
1,P12830,integrin_aEb7_complex,[CDH1],"[ITGAE, ITGB7]",Adhesion by Cadherin,CDH1-ITGAE+ITGB7
2,P12830,Q96E93,[CDH1],[KLRG1],Adhesion by Cadherin,CDH1-KLRG1
3,P19022,P19022,[CDH2],[CDH2],Adhesion by Cadherin,CDH2-CDH2
4,P19022,P06734,[CDH2],[FCER2],Adhesion by Cadherin,CDH2-FCER2


In [19]:
cell_options = sorted(marker_df["cell_label"].unique().tolist())
cell_options[:20]

['1-cell stage cell (Blastomere)',
 '4-cell stage cell (Blastomere)',
 '8-cell stage cell (Blastomere)',
 'A1 astrocyte',
 'A2 astrocyte',
 'AIRE-enriched cell',
 'AXL+ SIGLEC6+ dendritic cell',
 'AXL+SIGLEC6+ dendritic cell',
 'Abnormal myeloid cell',
 'Abnormal plasma cell',
 'Acinar cell',
 'Activated B cell',
 'Activated CD25+ regulatory T cell',
 'Activated CD4+ T cell',
 'Activated CD8+ T cell',
 'Activated T cell',
 'Activated cell',
 'Activated dendritic cell',
 'Activated double-positive T cell',
 'Activated effector cell']

In [21]:
search_a = widgets.Text(
    value="",
    placeholder="Search Cell A",
    description="Search A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

dropdown_a = widgets.Dropdown(
    options=cell_options,
    description="Cell A:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

search_b = widgets.Text(
    value="",
    placeholder="Search Cell B",
    description="Search B:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

dropdown_b = widgets.Dropdown(
    options=cell_options,
    description="Cell B:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

run_button = widgets.Button(
    description="Find possible pathways",
    button_style="danger"
)

output = widgets.Output()

In [23]:
def filter_dropdown(keyword, options):
    keyword = keyword.strip().lower()
    if keyword == "":
        return options
    return [x for x in options if keyword in x.lower()]

def update_a(change):
    filtered = filter_dropdown(change["new"], cell_options)
    if len(filtered) == 0:
        dropdown_a.options = ["No match found"]
        dropdown_a.value = "No match found"
    else:
        dropdown_a.options = filtered
        dropdown_a.value = filtered[0]

def update_b(change):
    filtered = filter_dropdown(change["new"], cell_options)
    if len(filtered) == 0:
        dropdown_b.options = ["No match found"]
        dropdown_b.value = "No match found"
    else:
        dropdown_b.options = filtered
        dropdown_b.value = filtered[0]

search_a.observe(update_a, names="value")
search_b.observe(update_b, names="value")

In [25]:
def get_cell_markers(cell_name):
    sub = marker_df[marker_df["cell_label"].str.lower() == str(cell_name).lower()].copy()
    genes = sorted(sub["Symbol"].dropna().astype(str).str.strip().unique().tolist())
    return set([g for g in genes if g != ""])

In [27]:
def find_possible_pathways(cell_a, cell_b):
    genes_a = get_cell_markers(cell_a)
    genes_b = get_cell_markers(cell_b)

    hits = []

    for _, row in interaction_clean.iterrows():
        row_genes_a = set(row["genes_a"])
        row_genes_b = set(row["genes_b"])

        forward_match = (len(genes_a.intersection(row_genes_a)) > 0) and (len(genes_b.intersection(row_genes_b)) > 0)
        reverse_match = (len(genes_a.intersection(row_genes_b)) > 0) and (len(genes_b.intersection(row_genes_a)) > 0)

        if forward_match or reverse_match:
            axis = f"{'/'.join(row['genes_a'])} ↔ {'/'.join(row['genes_b'])}"
            pathway = row["classification"] if row["classification"] != "" else "Unclassified"
            interactors = row["interactors"] if row["interactors"] != "" else axis

            hits.append({
                "axis": axis,
                "pathway": pathway,
                "interactors": interactors
            })

    hit_df = pd.DataFrame(hits).drop_duplicates()

    if len(hit_df) == 0:
        return hit_df, []

    pathway_list = sorted(hit_df["pathway"].dropna().unique().tolist())
    return hit_df, pathway_list

In [29]:
def run_tinder(b):
    with output:
        clear_output()

        cell_a = dropdown_a.value
        cell_b = dropdown_b.value

        if cell_a == "No match found" or cell_b == "No match found":
            print("Please select valid cell types.")
            return

        hit_df, pathway_list = find_possible_pathways(cell_a, cell_b)

        print(f"Cell A: {cell_a}")
        print(f"Cell B: {cell_b}")
        print("-" * 50)

        if len(hit_df) == 0:
            print("No possible pathways found.")
            return

        print("Possible pathways:")
        for p in pathway_list:
            print(f"- {p}")

        print("\nExample signaling axes:")
        for axis in hit_df["axis"].head(20).tolist():
            print(f"- {axis}")

run_button.on_click(run_tinder)

In [31]:
display(search_a)
display(dropdown_a)
display(search_b)
display(dropdown_b)
display(run_button)
display(output)

Text(value='', description='Search A:', layout=Layout(width='450px'), placeholder='Search Cell A', style=TextS…

Dropdown(description='Cell A:', layout=Layout(width='450px'), options=('1-cell stage cell (Blastomere)', '4-ce…

Text(value='', description='Search B:', layout=Layout(width='450px'), placeholder='Search Cell B', style=TextS…

Dropdown(description='Cell B:', layout=Layout(width='450px'), options=('1-cell stage cell (Blastomere)', '4-ce…

Button(button_style='danger', description='Find possible pathways', style=ButtonStyle())

Output()